# 05 — XGBoost Model Training

In [1]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

# --- Project Path Setup ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

if not os.path.exists("data/processed/machine_failure_features.csv"):
    raise FileNotFoundError(
        "Features dataset 'data/processed/machine_failure_features.csv' not found! "
        "Please run '03_feature_engineering.ipynb' first."
    )

# 1. Load the feature-engineered dataset
df = pd.read_csv("data/processed/machine_failure_features.csv")

# 2. Separate features (X) and target (y)
X = df.drop(columns=["Machine failure"])
y = df["Machine failure"]

# 3. Use the exact same train/test split as Random Forest
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Test set failure count:", y_test.sum(), "(expected 68)")

# 4. Calculate scale_pos_weight to handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Calculated scale_pos_weight: {scale_pos_weight:.4f}")

# 5. Instantiate and train XGBoost Classifier
# We pass X_train.values because XGBoost disallows '[' and ']' characters in feature names
model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=42, eval_metric="logloss"
)
model.fit(X_train.values, y_train)

# 6. Evaluate model on test set
pred = model.predict(X_test.values)
prob = model.predict_proba(X_test.values)[:, 1]

print("\n--- XGBoost Evaluation Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall:    {recall_score(y_test, pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, prob):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))

# 7. Feature Importance
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 5 Most Important Features:\n", importances.head())

# 8. Create models directory and save model
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/xgboost.pkl")

# Verify the file was created
xgb_saved = os.path.exists("models/xgboost.pkl")
print("\nFile exists check:", xgb_saved)
if not xgb_saved:
    raise FileNotFoundError("Failed to save 'models/xgboost.pkl'.")
print("Saved trained model to: models/xgboost.pkl")

Test set failure count: 68 (expected 68)
Calculated scale_pos_weight: 28.5203



--- XGBoost Evaluation Metrics ---
Accuracy:  0.9880
Precision: 0.8333
Recall:    0.8088
F1 Score:  0.8209
ROC-AUC:   0.9856

Confusion Matrix:
 [[1921   11]
 [  13   55]]

Top 5 Most Important Features:
 Rotational speed [rpm]         0.272328
Power [W]                      0.228707
Tool wear [min]                0.154627
Torque [Nm]                    0.133868
Wear-Torque product [minNm]    0.060708
dtype: float32

File exists check: True
Saved trained model to: models/xgboost.pkl
